# ANALISIS PENJUALAN HARIAN PT MENSANA ANEKA SATWA
## Data Analyst Test - 10 Cabang Distributor, 15 Produk, 3 Bulan (Jan-Mar 2026)

---

## SOAL

PT. Mensana memiliki 10 cabang distributor yang tersebar luas di seluruh Indonesia. 
Terdapat 15 produk unggulan yang dijual oleh masing-masing cabang distributor. Namun, 
data nama produk dan kode produk tiap cabang tidaklah sama (berbeda seluruhnya).  

### Yang harus dikerjakan:
1. Buatlah data mentah 3 bulan penjualan cabang (Kode Item, Nama Produk, kuantiti, omset penjualan per bulan)
2. Berikan dan buatlah solusi terkait kasus tersebut menggunakan data mentah hingga membentuk laporan yang siap dipresentasikan (Input, Proses, Output)
3. Laporan bertujuan untuk mengetahui:
   - Total omset seluruh cabang selama 3 bulan
   - Progress omset tiap cabang
   - Produk terlaris
   - Rekomendasi laporan tambahan
4. Buatlah Visualisasi data semaksimal mungkin
5. Waktu pengerjaan 24 jam

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('charts', exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

df_harian = pd.read_excel('Data_Mentah_Harian.xlsx', sheet_name='Data_Penjualan_Harian')
df_mapping = pd.read_excel('Data_Mentah_Harian.xlsx', sheet_name='Mapping_Produk')
df_harian['Tanggal'] = pd.to_datetime(df_harian['Tanggal'])

month_order = ['Januari', 'Februari', 'Maret']
day_order = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']

total_omset = df_harian['Omset'].sum()
total_qty = df_harian['Kuantiti'].sum()
hari_aktif = len(df_harian[df_harian['Kuantiti'] > 0]['Tanggal'].unique())

print(f'Total Records: {len(df_harian):,}')
print(f'Total Omset: Rp {total_omset:,.0f}')
print(f'Total Kuantiti: {total_qty:,} unit')

---
## 1. INPUT - Data Mentah

Data mentah yang sudah dibuat:
- **10 cabang distributor** di seluruh Indonesia
- **15 produk unggulan** per cabang
- **90 hari** (3 bulan: Januari - Maret 2026)
- Setiap cabang memiliki **kode dan nama produk yang berbeda-beda**

### Solusi Masalah Kode Produk Berbeda:
Dibuat **tabel mapping** yang menghubungkan kode produk standar dengan kode produk di masing-masing cabang.
Ini memungkinkan analisis konsolidasi di tingkat perusahaan.

In [ ]:
print('Contoh Mapping Produk:')
for _, row in df_mapping.head(3).iterrows():
    print(f'\nKode Standar: {row["Kode_Standar"]} - {row["Nama_Standar"]}')
    print(f'Kategori: {row["Kategori"]}')
    for cab in ['CAB-01', 'CAB-04', 'CAB-07']:
        print(f'  {cab}: {row[f"{cab}_Kode"]:15s} = {row[f"{cab}_Nama"]}')

---
## 2. PROSES - Transformasi & Mapping

### Masalah:
- Setiap cabang menggunakan **kode dan nama produk yang berbeda-beda**
- Contoh: Produk 'Supralit' di Medan bernama 'MED-V-001 / Supralit Anti Stress'
           Produk 'Supralit' di Jakarta bernama 'JKT-VA-001 / Supralit Powder'

### Solusi:
1. **Mapping Table**: Menghubungkan kode produk cabang dengan kode standar
2. **Konsolidasi**: Menggabungkan data dari semua cabang menggunakan kode standar
3. **Analisis**: Menghitung total omset, progress, dan produk terlaris

In [ ]:
df_bulanan = df_harian.groupby(['Bulan', 'ID_Cabang', 'Nama_Cabang', 'Provinsi', 
                                 'Kode_Standar', 'Nama_Standar', 'Kategori']).agg(
    Kuantiti=('Kuantiti', 'sum'),
    Omset=('Omset', 'sum')
).reset_index()
df_bulanan['Bulan'] = pd.Categorical(df_bulanan['Bulan'], categories=month_order, ordered=True)
print(f'Total records bulanan: {len(df_bulanan)}')

---
## 3. OUTPUT - Total Omset Seluruh Cabang

### 3.1 Ringkasan Total

In [ ]:
print('='*70)
print('OUTPUT 1: TOTAL OMSET SELURUH CABANG (3 BULAN)')
print('='*70)
print(f'\nTotal Omset 3 Bulan    : Rp {total_omset:>20,.0f}')
print(f'Total Kuantiti         : {total_qty:>20,} unit')
print(f'Hari Aktif Penjualan   : {hari_aktif:>20} hari')
print(f'Rata-rata Omset/Hari   : Rp {total_omset/hari_aktif:>20,.0f}')
print(f'Rata-rata Omset/Cabang : Rp {total_omset/10:>20,.0f}')

print('\n' + '-'*70)
print('RINGKASAN PER BULAN:')
print('-'*70)
monthly_summary = df_harian.groupby('Bulan').agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Hari=('Tanggal', 'nunique')
).reset_index()
monthly_summary['Bulan'] = pd.Categorical(monthly_summary['Bulan'], categories=month_order, ordered=True)
monthly_summary = monthly_summary.sort_values('Bulan')
monthly_summary['Rata_rata_Harian'] = (monthly_summary['Total_Omset'] / monthly_summary['Hari']).round(0)

for _, row in monthly_summary.iterrows():
    print(f"{row['Bulan']:10s}: Rp {row['Total_Omset']:>15,.0f} | {row['Total_Kuantiti']:>6,} unit | Rata2/Hari: Rp {row['Rata_rata_Harian']:>12,.0f}")

### 3.2 Progress Omset Tiap Cabang

In [ ]:
print('='*70)
print('OUTPUT 2: PROGRESS OMSET TIAP CABANG')
print('='*70)

progress = df_harian.groupby(['Nama_Cabang', 'Bulan']).agg(
    Omset=('Omset', 'sum'),
    Kuantiti=('Kuantiti', 'sum')
).reset_index()

progress_pivot = progress.pivot(index='Nama_Cabang', columns='Bulan', values='Omset')
progress_pivot = progress_pivot[['Januari', 'Februari', 'Maret']]
progress_pivot['Total_3_Bulan'] = progress_pivot.sum(axis=1)
progress_pivot['Growth_Jan_Feb_%'] = ((progress_pivot['Februari'] - progress_pivot['Januari']) / progress_pivot['Januari'] * 100).round(2)
progress_pivot['Growth_Feb_Mar_%'] = ((progress_pivot['Maret'] - progress_pivot['Februari']) / progress_pivot['Februari'] * 100).round(2)

print('\nPROGRESS OMSET TIAP CABANG (Rp):\n')
display(progress_pivot)

branch_ranking = df_harian.groupby(['ID_Cabang', 'Nama_Cabang']).agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Rata_rata_Harian=('Omset', 'mean')
).reset_index().sort_values('Total_Omset', ascending=False)

branch_ranking['Persentase'] = (branch_ranking['Total_Omset'] / total_omset * 100).round(2)
branch_ranking['Ranking'] = range(1, len(branch_ranking) + 1)

print('\nRANKING CABANG BERDASARKAN TOTAL OMSET:\n')
print('-'*90)
for _, row in branch_ranking.iterrows():
    print(f"#{int(row['Ranking']):2d} | {row['Nama_Cabang']:15s} | Rp {row['Total_Omset']:>15,.0f} | {row['Persentase']:5.1f}% | Rata2/Hari: Rp {row['Rata_rata_Harian']:>12,.0f}")

### 3.3 Produk Terlaris

In [ ]:
print('='*70)
print('OUTPUT 3: PRODUK TERLARIS (RANKING BY OMSET)')
print('='*70)

product_ranking = df_harian.groupby(['Nama_Standar', 'Kategori']).agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Rata_rata_Harian=('Omset', 'mean'),
    Jumlah_Cabang=('Nama_Cabang', 'nunique')
).reset_index().sort_values('Total_Omset', ascending=False)

product_ranking['Ranking'] = range(1, len(product_ranking) + 1)
product_ranking['Persentase'] = (product_ranking['Total_Omset'] / total_omset * 100).round(2)

print('\nRANKING 15 PRODUK TERLARIS:\n')
print('-'*100)
for _, row in product_ranking.iterrows():
    print(f"#{int(row['Ranking']):2d} | {row['Nama_Standar']:20s} | {row['Kategori']:15s} | Rp {row['Total_Omset']:>15,.0f} | {row['Total_Kuantiti']:>6,} unit | {row['Persentase']:5.1f}%")

print('\n' + '='*70)
print('TOP 5 PRODUK TERLARIS PER CABANG:')
print('='*70)

top5_per_branch = df_harian.groupby(['Nama_Cabang', 'Nama_Standar']).agg(
    Omset=('Omset', 'sum')
).reset_index()
top5_per_branch = top5_per_branch.sort_values(['Nama_Cabang', 'Omset'], ascending=[True, False])
top5_per_branch = top5_per_branch.groupby('Nama_Cabang').head(5)

for cabang in df_harian['Nama_Cabang'].unique():
    print(f'\n{cabang}:')
    subset = top5_per_branch[top5_per_branch['Nama_Cabang'] == cabang]
    for _, row in subset.iterrows():
        print(f"  - {row['Nama_Standar']:20s}: Rp {row['Omset']:>12,.0f}")

### 3.4 Analisis Kategori Produk

In [ ]:
print('='*70)
print('OUTPUT 4: ANALISIS KATEGORI PRODUK')
print('='*70)

category_summary = df_harian.groupby('Kategori').agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Jumlah_Produk=('Nama_Standar', 'nunique'),
    Rata_rata_Harian=('Omset', 'mean')
).reset_index().sort_values('Total_Omset', ascending=False)

category_summary['Persentase'] = (category_summary['Total_Omset'] / total_omset * 100).round(2)

print('\nDISTRIBUSI OMSET PER KATEGORI:\n')
print('-'*90)
for _, row in category_summary.iterrows():
    print(f"{row['Kategori']:20s} | Rp {row['Total_Omset']:>15,.0f} | {row['Persentase']:5.1f}% | {row['Jumlah_Produk']:2d} produk | Rata2/Hari: Rp {row['Rata_rata_Harian']:>12,.0f}")

### 3.5 Analisis Hari dalam Seminggu

In [ ]:
print('='*70)
print('OUTPUT 5: ANALISIS HARI DALAM SEMINGGU')
print('='*70)

dow_analysis = df_harian.groupby('Hari').agg(
    Total_Omset=('Omset', 'sum'),
    Rata_rata=('Omset', 'mean'),
    Jumlah_Transaksi=('Omset', 'count')
).reindex(day_order)

dow_analysis['Persentase'] = (dow_analysis['Total_Omset'] / total_omset * 100).round(2)

print('\nOMSET PER HARI DALAM SEMINGGU:\n')
print('-'*80)
for hari, row in dow_analysis.iterrows():
    print(f"{hari:10s} | Rp {row['Total_Omset']:>15,.0f} | {row['Persentase']:5.1f}% | Rata2: Rp {row['Rata_rata']:>12,.0f}")

---
## 4. ANALISIS LANJUTAN (Rekomendasi)

Berdasarkan data mentah yang tersedia, berikut analisis lanjutan untuk pengambilan keputusan:

### 4.1 Hari Kerja vs Akhir Pekan

In [ ]:
print('='*70)
print('ANALISIS 4.1: HARI KERJA vs AKHIR PEKAN')
print('='*70)

df_harian['Tipe_Hari'] = df_harian['Hari'].apply(
    lambda x: 'Hari Kerja' if x in ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat'] else 'Akhir Pekan'
)

tipe_hari_summary = df_harian.groupby('Tipe_Hari').agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Jumlah_Hari=('Tanggal', 'nunique'),
    Rata_rata_Harian=('Omset', 'mean')
).reset_index()

tipe_hari_summary['Persentase'] = (tipe_hari_summary['Total_Omset'] / total_omset * 100).round(2)

print('\nPERBANDINGAN HARI KERJA vs AKHIR PEKAN:\n')
for _, row in tipe_hari_summary.iterrows():
    print(f"{row['Tipe_Hari']:15s} | Omset: Rp {row['Total_Omset']:>15,.0f} | {row['Persentase']:5.1f}% | Hari: {int(row['Jumlah_Hari'])} | Rata2/Hari: Rp {row['Rata_rata_Harian']:>12,.0f}")

fig_weekday = px.bar(
    tipe_hari_summary,
    x='Tipe_Hari',
    y='Rata_rata_Harian',
    color='Tipe_Hari',
    title='Perbandingan Rata-rata Omset Harian: Hari Kerja vs Akhir Pekan',
    labels={'Rata_rata_Harian': 'Rata-rata Omset/Hari (Rp)', 'Tipe_Hari': 'Tipe Hari'},
    text_auto=':.0f'
)
fig_weekday.update_layout(showlegend=False, height=400)
fig_weekday.show()

### 4.2 Dampak Hari Libur Nasional

In [ ]:
print('='*70)
print('ANALISIS 4.2: DAMPAK HARI LIBUR NASIONAL')
print('='*70)

holiday_dates = {
    '2026-01-01': 'Tahun Baru',
    '2026-01-29': "Isra Mi'raj",
    '2026-02-17': 'Tahun Baru Imlek',
    '2026-03-19': 'Nyepi'
}

df_harian['Tanggal_str'] = df_harian['Tanggal'].dt.strftime('%Y-%m-%d')
df_harian['Hari_Libur'] = df_harian['Tanggal_str'].map(holiday_dates)
df_harian['Hari_Libur'] = df_harian['Hari_Libur'].fillna('Hari Normal')

holiday_daily = df_harian.groupby(['Tanggal', 'Hari_Libur']).agg(
    Total_Omset=('Omset', 'sum')
).reset_index()

holiday_impact = holiday_daily.groupby('Hari_Libur').agg(
    Rata_rata_Omset=('Total_Omset', 'mean'),
    Jumlah_Hari=('Tanggal', 'nunique')
).reset_index()

normal_avg = holiday_impact[holiday_impact['Hari_Libur'] == 'Hari Normal']['Rata_rata_Omset'].values[0]
holiday_impact['Deviasi_%'] = ((holiday_impact['Rata_rata_Omset'] - normal_avg) / normal_avg * 100).round(2)

print('\nDAMPAK HARI LIBUR TERHADAP OMSET:\n')
print('-'*70)
print(f"{'Hari':20s} | {'Rata-rata Omset':>18s} | {'Deviasi vs Normal':>18s}")
print('-'*70)
for _, row in holiday_impact.iterrows():
    sign = '+' if row['Deviasi_%'] > 0 else ''
    print(f"{row['Hari_Libur']:20s} | Rp {row['Rata_rata_Omset']:>15,.0f} | {sign}{row['Deviasi_%']:.1f}%")

### 4.3 Mix Produk per Cabang

In [ ]:
print('='*70)
print('ANALISIS 4.3: MIX PRODUK PER CABANG')
print('='*70)

mix_data = df_harian.groupby(['Nama_Cabang', 'Kategori']).agg(
    Omset=('Omset', 'sum')
).reset_index()

mix_total = mix_data.groupby('Nama_Cabang')['Omset'].sum().reset_index()
mix_total.columns = ['Nama_Cabang', 'Total_Omset']
mix_data = mix_data.merge(mix_total, on='Nama_Cabang')
mix_data['Persentase'] = (mix_data['Omset'] / mix_data['Total_Omset'] * 100).round(2)

fig_mix = px.bar(
    mix_data,
    x='Nama_Cabang',
    y='Persentase',
    color='Kategori',
    title='Persentase Kategori Produk per Cabang',
    labels={'Persentase': 'Persentase (%)', 'Nama_Cabang': 'Cabang'},
    barmode='stack',
    height=500
)
fig_mix.update_layout(xaxis_tickangle=-45)
fig_mix.show()

### 4.4 Efisiensi Harga

In [ ]:
print('='*70)
print('ANALISIS 4.4: EFISIENSI HARGA JUAL vs HARGA STANDAR')
print('='*70)

price_analysis = df_harian[df_harian['Kuantiti'] > 0].groupby(['Nama_Cabang', 'Nama_Standar']).agg(
    Harga_Rata_rata=('Harga_Satuan', 'mean'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Total_Omset=('Omset', 'sum')
).reset_index()

price_mapping = df_mapping[['Nama_Standar', 'Harga_Standar']].drop_duplicates()
price_analysis = price_analysis.merge(price_mapping, on='Nama_Standar', how='left')
price_analysis['Deviasi_Harga_%'] = ((price_analysis['Harga_Rata_rata'] - price_analysis['Harga_Standar']) / price_analysis['Harga_Standar'] * 100).round(2)

branch_price = price_analysis.groupby('Nama_Cabang').agg(
    Deviasi_Rata_rata=('Deviasi_Harga_%', 'mean')
).reset_index().sort_values('Deviasi_Rata_rata')

print('\nDEVIASI HARGA JUAL vs STANDAR PER CABANG:\n')
for _, row in branch_price.iterrows():
    sign = '+' if row['Deviasi_Rata_rata'] > 0 else ''
    print(f"  {row['Nama_Cabang']:15s}: {sign}{row['Deviasi_Rata_rata']:.2f}%")

fig_price = px.bar(
    branch_price,
    x='Nama_Cabang',
    y='Deviasi_Rata_rata',
    color='Deviasi_Rata_rata',
    color_continuous_scale=['red', 'yellow', 'green'],
    color_continuous_midpoint=0,
    title='Deviasi Harga Jual vs Harga Standar per Cabang',
    labels={'Deviasi_Rata_rata': 'Deviasi (%)', 'Nama_Cabang': 'Cabang'}
)
fig_price.update_layout(showlegend=False, height=400)
fig_price.show()

### 4.5 Tren Harian dengan Moving Average

In [ ]:
print('='*70)
print('ANALISIS 4.5: TREN HARIAN DENGAN MOVING AVERAGE')
print('='*70)

daily_total = df_harian.groupby('Tanggal').agg(
    Total_Omset=('Omset', 'sum')
).reset_index().sort_values('Tanggal')

daily_total['MA_7_Hari'] = daily_total['Total_Omset'].rolling(window=7, min_periods=1).mean()
daily_total['MA_14_Hari'] = daily_total['Total_Omset'].rolling(window=14, min_periods=1).mean()

fig_ma = go.Figure()
fig_ma.add_trace(go.Scatter(
    x=daily_total['Tanggal'], y=daily_total['Total_Omset'],
    name='Omset Harian', mode='lines', line=dict(color='lightblue', width=1)
))
fig_ma.add_trace(go.Scatter(
    x=daily_total['Tanggal'], y=daily_total['MA_7_Hari'],
    name='MA 7 Hari', mode='lines', line=dict(color='orange', width=2)
))
fig_ma.add_trace(go.Scatter(
    x=daily_total['Tanggal'], y=daily_total['MA_14_Hari'],
    name='MA 14 Hari', mode='lines', line=dict(color='red', width=2)
))

fig_ma.update_layout(
    title='Tren Omset Harian dengan Moving Average',
    xaxis_title='Tanggal',
    yaxis_title='Omset (Rp)',
    height=500,
    legend=dict(x=0, y=1)
)
fig_ma.show()

### 4.6 Alert Penjualan Abnormal

In [ ]:
print('='*70)
print('ANALISIS 4.6: ALERT PENJUALAN ABNORMAL')
print('='*70)

daily_total['Mean'] = daily_total['Total_Omset'].mean()
daily_total['Std'] = daily_total['Total_Omset'].std()
daily_total['Upper_Threshold'] = daily_total['Mean'] + 2 * daily_total['Std']
daily_total['Lower_Threshold'] = daily_total['Mean'] - 2 * daily_total['Std']
daily_total['Abnormal'] = (
    (daily_total['Total_Omset'] > daily_total['Upper_Threshold']) |
    (daily_total['Total_Omset'] < daily_total['Lower_Threshold'])
)

abnormal_days = daily_total[daily_total['Abnormal']].copy()
abnormal_days['Hari'] = abnormal_days['Tanggal'].dt.day_name()

print(f'\nThreshold Atas : Rp {daily_total["Upper_Threshold"].iloc[0]:,.0f}')
print(f'Threshold Bawah: Rp {daily_total["Lower_Threshold"].iloc[0]:,.0f}')
print(f'Hari Abnormal  : {len(abnormal_days)} dari {len(daily_total)} hari\n')

if len(abnormal_days) > 0:
    print('DETAIL HARI ABNORMAL:')
    print('-'*60)
    for _, row in abnormal_days.iterrows():
        status = 'TINGGI' if row['Total_Omset'] > row['Upper_Threshold'] else 'RENDAH'
        print(f"  {row['Tanggal'].strftime('%Y-%m-%d')} ({row['Hari']:10s}): Rp {row['Total_Omset']:>12,.0f} [{status}]")

fig_anomaly = go.Figure()
fig_anomaly.add_trace(go.Scatter(
    x=daily_total['Tanggal'], y=daily_total['Total_Omset'],
    name='Omset Harian', mode='lines', line=dict(color='steelblue', width=1.5)
))
fig_anomaly.add_trace(go.Scatter(
    x=daily_total['Tanggal'], y=daily_total['Upper_Threshold'],
    name='Upper Threshold (+2 std)', mode='lines', line=dict(color='red', width=1.5, dash='dash')
))
fig_anomaly.add_trace(go.Scatter(
    x=daily_total['Tanggal'], y=daily_total['Lower_Threshold'],
    name='Lower Threshold (-2 std)', mode='lines', line=dict(color='red', width=1.5, dash='dash')
))
fig_anomaly.add_trace(go.Scatter(
    x=abnormal_days['Tanggal'], y=abnormal_days['Total_Omset'],
    name='Abnormal', mode='markers', marker=dict(color='red', size=10, symbol='x')
))

fig_anomaly.update_layout(
    title='Alert: Hari dengan Penjualan Abnormal (>2 Std Dev)',
    xaxis_title='Tanggal',
    yaxis_title='Omset (Rp)',
    height=500
)
fig_anomaly.show()

### 4.7 Proyeksi 1 Bulan ke Depan (dengan Trendline)

In [ ]:
# Proyeksi 1 bulan ke depan dengan garis regresi lurus
daily_omset = df_harian.groupby('Tanggal')['Omset'].sum().reset_index()
daily_omset['day_num'] = (daily_omset['Tanggal'] - daily_omset['Tanggal'].min()).dt.days

coeffs = np.polyfit(daily_omset['day_num'], daily_omset['Omset'], 1)
trend_fn = np.poly1d(coeffs)

last_date = daily_omset['Tanggal'].max()
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30)
forecast_day_nums = np.array([(d - daily_omset['Tanggal'].min()).days for d in forecast_dates])
forecast_values = trend_fn(forecast_day_nums)

monthly_omset = df_harian.groupby('Bulan')['Omset'].sum()
apr_proj = forecast_values.sum()

# Garis regresi lurus: nilai harian x jumlah hari per bulan
month_start_days = {'Januari': 0, 'Februari': 31, 'Maret': 59, 'April': 90}
month_n_days = {'Januari': 31, 'Februari': 28, 'Maret': 31, 'April': 30}

months = ['Januari', 'Februari', 'Maret', 'April\n(Proyeksi)']
bar_heights = [monthly_omset['Januari'], monthly_omset['Februari'], monthly_omset['Maret'], apr_proj]
trend_straight = [trend_fn(month_start_days[m]) * month_n_days[m] for m in ['Januari', 'Februari', 'Maret', 'April']]

print('='*70)
print('PROYEKSI PENJUALAN 1 BULAN KE DEPAN (April 2026)')
print('='*70)
print(f'\nJanuari 2026  : Rp {monthly_omset["Januari"]:>15,.0f}')
print(f'Februari 2026 : Rp {monthly_omset["Februari"]:>15,.0f}')
print(f'Maret 2026    : Rp {monthly_omset["Maret"]:>15,.0f}')
print(f'April 2026    : Rp {apr_proj:>15,.0f} (proyeksi)')
print(f'\nTrend Slope   : +Rp {coeffs[0]:,.0f}/hari')

colors = ['#636EFA', '#636EFA', '#636EFA', '#00CC96']

fig_proj = go.Figure()
fig_proj.add_trace(go.Bar(
    x=months, y=bar_heights, name='Omset',
    marker_color=colors,
    text=[f'Rp {v:,.0f}' for v in bar_heights],
    textposition='outside'
))
fig_proj.add_trace(go.Scatter(
    x=months, y=trend_straight, name='Garis Regresi',
    mode='lines',
    line=dict(color='#EF553B', width=3)
))
fig_proj.update_layout(
    title='PROYEKSI 1 BULAN KE DEPAN (April 2026) + Garis Regresi',
    xaxis_title='Bulan', yaxis_title='Omset (Rp)',
    height=450, legend=dict(x=0, y=1),
    yaxis=dict(rangemode='tozero')
)
fig_proj.show()
fig_proj.write_image('charts/09_proyeksi_april.png', scale=2)

---
## 5. VISUALISASI (5 Chart Utama)

In [ ]:
# VIS 1: Total Omset per Cabang (Horizontal Bar)
fig1 = px.bar(
    branch_ranking,
    y='Nama_Cabang',
    x='Total_Omset',
    color='Nama_Cabang',
    orientation='h',
    title='1. TOTAL OMSET PER CABANG (3 BULAN)',
    labels={'Total_Omset': 'Total Omset (Rp)', 'Nama_Cabang': 'Cabang'},
    text_auto=':.2s'
)
fig1.update_layout(showlegend=False, height=450, yaxis={'categoryorder':'total ascending'})
fig1.show()
fig1.write_image('charts/01_omset_cabang_bulanan.png', scale=2)

In [ ]:
# VIS 2: Tren Harian Seluruh Cabang (Line Chart)
daily_total_plot = df_harian.groupby('Tanggal').agg(
    Total_Omset=('Omset', 'sum')
).reset_index()

fig2 = px.line(
    daily_total_plot,
    x='Tanggal',
    y='Total_Omset',
    title='2. TREN OMSET HARIAN SELURUH CABANG',
    labels={'Total_Omset': 'Omset (Rp)', 'Tanggal': 'Tanggal'},
    markers=False
)
fig2.update_layout(height=400)
fig2.show()
fig2.write_image('charts/05_tren_harian.png', scale=2)

In [ ]:
# VIS 3: Hari Kerja vs Akhir Pekan
df_harian['Tipe_Hari'] = df_harian['Hari'].apply(
    lambda x: 'Akhir Pekan' if x in ['Sabtu', 'Minggu'] else 'Hari Kerja'
)
dow_compare = df_harian.groupby('Tipe_Hari')['Omset'].mean().reset_index()
dow_compare.columns = ['Tipe_Hari', 'Rata_rata_Omset']

fig3 = px.bar(
    dow_compare,
    x='Tipe_Hari',
    y='Rata_rata_Omset',
    color='Tipe_Hari',
    title='3. RATA-RATA OMSET: HARI KERJA vs AKHIR PEKAN',
    labels={'Rata_rata_Omset': 'Rata-rata Omset (Rp)', 'Tipe_Hari': 'Tipe Hari'},
    text_auto=':.0f'
)
fig3.update_layout(showlegend=False, height=400)
fig3.show()
fig3.write_image('charts/06_hari_kerja_weekend.png', scale=2)

In [ ]:
# VIS 4: Dampak Hari Libur Nasional
holiday_names = {
    '2026-01-01': 'Tahun Baru',
    '2026-01-29': "Isra Mi'raj",
    '2026-02-17': 'Tahun Baru Imlek',
    '2026-03-19': 'Nyepi'
}

holiday_labels = ['Hari Normal']
holiday_values = [df_harian[~df_harian['Tanggal'].dt.strftime('%Y-%m-%d').isin(holiday_names.keys())]['Omset'].mean()]

for date_str, name in holiday_names.items():
    date = pd.Timestamp(date_str)
    if date in df_harian['Tanggal'].values:
        holiday_labels.append(name)
        holiday_values.append(df_harian[df_harian['Tanggal'] == date]['Omset'].sum())

fig4 = px.bar(
    x=holiday_labels,
    y=holiday_values,
    title='4. DAMPAK HARI LIBUR NASIONAL TERHADAP OMSET',
    labels={'x': 'Hari', 'y': 'Omset (Rp)'},
    text_auto=':.2s',
    color=holiday_labels,
    color_discrete_sequence=['#636EFA'] + ['#EF553B'] * len(holiday_names)
)
fig4.update_layout(showlegend=False, height=400)
fig4.show()
fig4.write_image('charts/08_dampak_libur.png', scale=2)

In [ ]:
# VIS 5: Top 10 Produk Terlaris (Horizontal Bar)
fig5 = px.bar(
    product_ranking.head(10),
    x='Total_Omset',
    y='Nama_Standar',
    color='Kategori',
    title='5. TOP 10 PRODUK TERLARIS (by Omset)',
    labels={'Total_Omset': 'Total Omset (Rp)', 'Nama_Standar': 'Produk'},
    orientation='h',
    text_auto=':.2s'
)
fig5.update_layout(height=450, yaxis={'categoryorder':'total ascending'})
fig5.show()
fig5.write_image('charts/03_top10_produk.png', scale=2)

---
## 6. KESIMPULAN & REKOMENDASI

### Temuan Utama:
1. **Total omset 3 bulan**: Rp 2,253,861,298
2. **Jakarta** menjadi cabang dengan omset tertinggi
3. **Produk Vitamin** mendominasi penjualan (~39%)
4. **Februari** mengalami penurunan (seasonal)
5. **Maret** menunjukkan recovery yang baik

### Rekomendasi:
1. **Tingkatkan penjualan** di cabang Pontianak dan Denpasar (perlu analisis lebih dalam)
2. **Optimalkan mix produk** di setiap cabang berdasarkan tren penjualan
3. **Manfaatkan tren positif** di bulan Maret untuk momentum Q2
4. **Perkuat distribusi** produk Antibiotik yang menunjukkan potensi besar
5. **Review pricing strategy** untuk memastikan harga jual optimal vs standar

In [ ]:
print('='*70)
print('ANALISIS SELESAI!')
print('='*70)
print('File: Analisis_Harian.ipynb')
print('Data: Data_Mentah_Harian.xlsx')